In [ ]:
!pip install easyocr

In [27]:
# 1. 설치

import cv2
import numpy as np
import re
import easyocr

# --- 1. EasyOCR 초기화 ---
# gpu=True: T4 GPU 사용 (속도 향상), False: CPU 사용
reader = easyocr.Reader(['ko', 'en'], gpu=True)

# --- 2. 이미지 설정 ---
image_path = "unnamed.jpg"

# --- 3. 텍스트 정제 및 필터링 ---

def clean_text(text):
    """OCR 오인식 교정"""
    # EasyOCR도 가끔 특수문자 노이즈가 생기므로 제거
    text = re.sub(r"[|│]", "", text)
    return text.strip()

def is_noise(text):
    """시스템 메시지 등 노이즈 제거"""
    system_patterns = [
        r".*님이 들어왔습니다\.$", r".*님이 나갔습니다\.$",
        r".*님이 .*님을 초대했습니다\.$", r"^삭제된 메시지입니다\.$",
        r".*기프티콘을 보냈습니다.*", r".*송금했습니다.*", r"^톡게시판.*"
    ]
    for pattern in system_patterns:
        if re.match(pattern, text): return True

    ui_keywords = {
        "사진", "동영상", "음성메시지", "보이스톡", "페이스톡", "라이브톡",
        "선물하기", "송금", "정산하기", "프로필 보기", "공지 등록", "좋아요", "공감",
        "안읽음", "MY"
    }
    if text in ui_keywords: return True
    # 숫자만 있는데 길이가 짧으면 (안 읽은 사람 수 '1') 제거
    if text.replace(':', '').isdigit() and len(text) < 3: return True
    return False

def format_time(ts_str):
    """시간 포맷 통일 (오전/오후 -> 24시간제 변환, 콜론 없는 경우 포함)"""
    ts_str = ts_str.replace(" ", "")
    final_time = ts_str

    try:
        is_pm = "오후" in ts_str
        is_am = "오전" in ts_str

        # [수정] 1순위: 콜론(:)이나 점(.)이 있는 경우 (예: 4:06, 4.06)
        match = re.search(r"(\d{1,2})[:\.\,](\d{2})", ts_str)

        # [추가] 2순위: 구분자가 없지만 '오전/오후'가 있고 숫자 3~4자리인 경우 (예: 오후406)
        if not match and (is_pm or is_am):
            match = re.search(r"(\d{1,2})(\d{2})$", ts_str)

        if match:
            hour, minute = int(match.group(1)), int(match.group(2))

            if is_pm and hour != 12: hour += 12
            if is_am and hour == 12: hour = 0

            final_time = f"{hour:02}:{minute:02}"

    except: pass
    return final_time

# --- 4. OCR 실행 및 데이터 변환 ---

# EasyOCR 결과: [[bbox, text, conf], ...]
result = reader.readtext(image_path)

processed_items = []

# 정규식
time_regex = re.compile(r".*[\d]{1,2}:[\d]{2}$") # "숫자:숫자"로 끝나는 패턴
time_regex2 = re.compile(r".*[\d]{1,2}.[\d]{2}$") # "숫자.숫자"로 끝나는 패턴
time_regex4 = re.compile(r".*[\d]{1,2},[\d]{2}$") # "숫자,숫자"로 끝나는 패턴
time_regex3 = re.compile(r".*오[전후]\s*\d{3,4}$")
date_regex = re.compile(r"^20\d{2}. \d{1,2}. \d{1,2}.*") # 날짜 패턴

if not result:
    print("텍스트를 찾을 수 없습니다.")
else:
    # 이미지 너비 확인
    img = cv2.imread(image_path)
    if img is None:
        # 이미지를 못 읽었을 경우 대비 (EasyOCR은 이미지 파일 자체를 읽음)
        from PIL import Image
        with Image.open(image_path) as pil_img:
            width, height = pil_img.size
            IMAGE_WIDTH = width
    else:
        IMAGE_WIDTH = img.shape[1]

    center_x = IMAGE_WIDTH / 2

    for item in result:
        box = item[0]   # [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]
        text = item[1]  # 텍스트

        text = clean_text(text)
        if is_noise(text): continue

        # 좌표 변환 (리스트 -> numpy -> 중심/범위 계산)
        np_box = np.array(box)
        y_center = (np.min(np_box[:, 1]) + np.max(np_box[:, 1])) / 2
        x_left = np.min(np_box[:, 0])
        x_right = np.max(np_box[:, 0])

        # 타입 판별
        is_timestamp = bool(time_regex.match(text)) or bool(time_regex2.match(text)) or bool(time_regex3.match(text)) or bool(time_regex4.match(text))
        is_date = bool(date_regex.match(text))

        processed_items.append({
            'text': text, 'y_center': y_center, 'x_left': x_left, 'x_right': x_right,
            'is_timestamp': is_timestamp, 'is_date': is_date
        })

    # --- 5. 줄 그룹화 (Y좌표 기준) ---
    processed_items.sort(key=lambda x: x['y_center'])
    Y_TOLERANCE = 20
    grouped_lines = []

    if processed_items:
        curr_items = [processed_items[0]]
        curr_y = processed_items[0]['y_center']

        for item in processed_items[1:]:
            if abs(item['y_center'] - curr_y) < Y_TOLERANCE:
                curr_items.append(item)
            else:
                # 줄 바꿈 발생 시 저장
                texts = sorted([i for i in curr_items if not i['is_timestamp'] and not i['is_date']], key=lambda x: x['x_left'])
                times = sorted([i for i in curr_items if i['is_timestamp']], key=lambda x: x['x_left'])
                dates = sorted([i for i in curr_items if i['is_date']], key=lambda x: x['x_left'])

                if texts: grouped_lines.append({'type': 'text', 'items': texts})
                if times: grouped_lines.append({'type': 'timestamp', 'items': times})
                if dates: grouped_lines.append({'type': 'date', 'items': dates})

                curr_items = [item]
                curr_y = item['y_center']

        # 마지막 줄 저장
        texts = sorted([i for i in curr_items if not i['is_timestamp'] and not i['is_date']], key=lambda x: x['x_left'])
        times = sorted([i for i in curr_items if i['is_timestamp']], key=lambda x: x['x_left'])
        dates = sorted([i for i in curr_items if i['is_date']], key=lambda x: x['x_left'])
        if texts: grouped_lines.append({'type': 'text', 'items': texts})
        if times: grouped_lines.append({'type': 'timestamp', 'items': times})
        if dates: grouped_lines.append({'type': 'date', 'items': dates})

from datetime import datetime, timedelta

# --- 6. 최종 로그 변환 (버퍼링 로직) ---

# [추가됨] 1. 날짜 리스트 프리 스캔 (Pre-scan)
detected_dates = []
for line in grouped_lines:
    if line['type'] == 'date':
        # 텍스트 합치기
        text = " ".join([i['text'] for i in line['items']])
        # 날짜 추출 정규식 (연, 월, 일)
        match = re.search(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일", text)
        if match:
            y, m, d = map(int, match.groups())
            detected_dates.append(datetime(y, m, d))

# [추가됨] 2. 초기 날짜(날짜 미상 대체값) 계산
if detected_dates:
    earliest_date = min(detected_dates)       # 가장 빠른 날짜 찾기
    start_date = earliest_date - timedelta(days=1) # 하루 전 날짜 계산
    # 초기 current_date 설정 (예: 2024. 6. 15.)
    current_date = f"{start_date.year}. {start_date.month}. {start_date.day}."
else:
    current_date = "2000. 1. 1."


# 3. 메인 변환 루프
final_chat = []
current_turn_lines = []
# current_date는 위에서 이미 설정됨

for line in grouped_lines:
    if line['type'] == 'date':
        raw_date = " ".join([i['text'] for i in line['items']])
        # 정규식으로 연, 월, 일 숫자만 추출해서 업데이트
        date_match = re.search(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일", raw_date)
        if date_match:
            current_date = f"{date_match.group(1)}. {date_match.group(2)}. {date_match.group(3)}."
        else:
            # 포맷이 안 맞을 경우 원본 유지하거나 이전 날짜 유지
            pass

    elif line['type'] == 'text':
        current_turn_lines.append(line)

    elif line['type'] == 'timestamp':
        if not current_turn_lines: continue

        # 타임스탬프 텍스트 변환
        raw_time = " ".join([i['text'] for i in line['items']])
        time_str = format_time(raw_time)

        # 날짜와 시간 결합
        full_ts = f"{current_date} {time_str}"

        # 발화자 판단 로직
        first_line = current_turn_lines[0]
        first_line_min_x = first_line['items'][0]['x_left']
        first_line_max_x = first_line['items'][-1]['x_right']
        first_center = (first_line_min_x + first_line_max_x) / 2

        speaker = ""
        messages = []

        if first_center > center_x:
            speaker = "나"
            for l in current_turn_lines:
                messages.append(" ".join([i['text'] for i in l['items']]))
        else:
            speaker = " ".join([i['text'] for i in first_line['items']])
            for l in current_turn_lines[1:]:
                messages.append(" ".join([i['text'] for i in l['items']]))

        full_message = " ".join(messages)
        if full_message:
            final_chat.append(f"{full_ts}, {speaker} : {full_message}")

        current_turn_lines = []

S=''

# 출력
for line in final_chat:
    S+=line+'\n'

In [ ]:
final_chat

In [ ]:
import os
import json
import vertexai
from vertexai.generative_models import GenerativeModel

def run_gemini_flash(json_key_path, prompt_text):
    """
    JSON 키 파일을 사용하여 인증하고 Gemini 모델을 실행하는 함수
    """

    # 1. JSON 파일 존재 확인
    if not os.path.exists(json_key_path):
        raise FileNotFoundError(f"키 파일을 찾을 수 없습니다: {json_key_path}")

    # 2. JSON 파일에서 프로젝트 ID 추출 (Service Account JSON 구조 가정)
    try:
        with open(json_key_path, 'r') as f:
            key_data = json.load(f)
            project_id = key_data.get("project_id")

        if not project_id:
            raise ValueError("JSON 파일에 'project_id' 정보가 없습니다.")

    except Exception as e:
        print(f"JSON 파일 읽기 오류: {e}")
        return

    # 3. 인증 환경 변수 설정 (Google Cloud 인증 표준 방식)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = json_key_path

    # 4. Vertex AI 초기화
    # location은 리전(Region)을 의미합니다. (예: us-central1, asia-northeast3 등)
    vertexai.init(project=project_id, location="us-central1")

    # 5. 모델 로드
    # 주의: Gemini 2.5 Flash가 출시되면 모델 이름을 해당 버전으로 변경하세요.
    # 예: "gemini-2.5-flash-preview" 또는 공식 명칭
    model_name = "gemini-2.5-flash"

    try:
        model = GenerativeModel(model_name)

        # 6. 콘텐츠 생성 (답변 받기)
        print(f"--- 입력 문장: {prompt_text} ---")
        response = model.generate_content(prompt_text)

        return response.text

    except Exception as e:
        return f"모델 실행 중 오류 발생: {e}"

# --- 실행 예시 ---
if __name__ == "__main__":
    # 사용자의 JSON 파일 경로를 입력하세요
    my_key_path = "/content/black-radius-435710-t2-13e26d172fa3.json"

    # 입력할 문장
    input_sentence = S+'위의 문장은 OCR을 진행한 결과로, 이 대화에서 OCR이 오류로 판단한 문자를 정정하는 사전 처리 과정을 진행해줘. 특히 ㅎㅎ가 층, 등, 승 으로 나오는 "초성 오해 문제"를 예의주시하렴. 사전 처리 과정이 끝나면 타임스탬프와 발화자 이름을 제외한 나머지 부분을 기준으로, 틀린 글자가 전체에서 차지하는 비율을 출력해 줘'

    prompt = '\n너는 약속 장소 결정 전문 컨설턴트로, 방금 사전처리한 결과 약속 장소를 정하기 위해 필요한 요소인 성격, 선호하는 것과 선호하지 않는 것, 만나기 위한 위치 등을 출력해 줘. 이 응답은 최종 약속 장소와 코스를 정하기 위한 프롬프트의 입력으로 들어갈 거야'

    # 함수 실행
    result = run_gemini_flash(my_key_path, input_sentence+prompt)

    print("\n[Gemini 응답 결과]")
    print(result)